# Study 1, Amendment 1: prompt robustness (A2) and auditability of CLEAR's explanations (A1)

Analyses fixed in the OSF amendment dated 3 October 2026, run on the same NIH ChestX-ray14 cohort as the registered analysis.

**Settings:** Accelerator **GPU T4 x2**, Internet **On**, inputs **NIH Chest X-rays** and your private **chexzero-weights** dataset.
**Run:** Save Version, then Save & Run All (Commit). About 1.5 hours. At the end copy the block between `===== AMENDMENT SUMMARY =====` and `===== END =====`.

In [ ]:
# 0. Settings
import os, sys, time, json, glob, subprocess, hashlib, platform
os.environ["HF_HOME"] = "/tmp/hf"          # large downloads go to /tmp, not the 20 GB output folder
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
SEED = 2026
OUT = "/kaggle/working/amend1_out"; os.makedirs(OUT, exist_ok=True)
FINDINGS = ["Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule","Pneumonia",
            "Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis","Pleural_Thickening","Hernia"]
PROMPT_WORDS = {"Atelectasis":"atelectasis","Cardiomegaly":"cardiomegaly","Effusion":"pleural effusion",
                "Infiltration":"infiltration","Mass":"mass","Nodule":"nodule","Pneumonia":"pneumonia",
                "Pneumothorax":"pneumothorax","Consolidation":"consolidation","Edema":"edema",
                "Emphysema":"emphysema","Fibrosis":"fibrosis","Pleural_Thickening":"pleural thickening",
                "Hernia":"hernia"}
POS = [PROMPT_WORDS[f] for f in FINDINGS]
NEG = ["no " + w for w in POS]
run_info = {"started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), "seed": SEED,
            "prompts_pos": POS, "prompts_neg": NEG}
T0 = time.time()
TEMPLATES = {"T1": ("{f}", "no {f}"),
             "T2": ("chest x-ray showing {f}", "chest x-ray showing no {f}"),
             "T3": ("there is {f}", "there is no {f}"),
             "T4": ("findings consistent with {f}", "no evidence of {f}")}
TP_TXT = {t: [p.format(f=w) for w in POS] for t, (p, n) in TEMPLATES.items()}
TN_TXT = {t: [n.format(f=w) for w in POS] for t, (p, n) in TEMPLATES.items()}
run_info["templates"] = TEMPLATES
RELEVANCE = {"Atelectasis": r"atelecta", "Cardiomegaly": r"cardiomegal|heart size|enlarged heart|cardiac silhouette|cardiac enlargement|heart is enlarged|cardiac size",
  "Effusion": r"effusion", "Infiltration": r"infiltrat", "Mass": r"\bmass(es)?\b", "Nodule": r"nodul", "Pneumonia": r"pneumonia",
  "Pneumothorax": r"pneumothora", "Consolidation": r"consolidat", "Edema": r"edema|oedema", "Emphysema": r"emphysem",
  "Fibrosis": r"fibros|fibrotic", "Pleural_Thickening": r"pleural thickening|thickened pleura", "Hernia": r"hernia"}

In [ ]:
# 1. Packages and versions
!pip install -q git+https://github.com/openai/CLIP.git gdown
!rm -rf /kaggle/working/CLEAR && git clone -q https://github.com/peterhan91/CLEAR.git /kaggle/working/CLEAR
!pip install -q /kaggle/working/CLEAR
import torch, numpy as np, pandas as pd, transformers
run_info["clear_git_commit"] = subprocess.run(["git","-C","/kaggle/working/CLEAR","rev-parse","HEAD"],
                                              capture_output=True, text=True).stdout.strip()
try:
    from huggingface_hub import HfApi
    run_info["clear_hf_sha"] = HfApi().model_info("peterhan91/CLEAR").sha
except Exception as e:
    run_info["clear_hf_sha"] = "unavailable: " + repr(e)[:200]
run_info.update({"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__,
                 "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
print(json.dumps(run_info, indent=1))
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2"

In [ ]:
# 2. Cohort: official test split + 10% held-out training patients
def find_file(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if not d.startswith("images_")]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)
entry_csv = find_file("Data_Entry_2017.csv"); base = os.path.dirname(entry_csv)
img_paths = {os.path.basename(p): p for p in glob.glob(os.path.join(base, "images_*", "images", "*.png"))}
print("images:", len(img_paths))

df = pd.read_csv(entry_csv).rename(columns={"Image Index":"image","Finding Labels":"labels",
        "Patient ID":"patient","Patient Age":"age","Patient Gender":"sex","View Position":"view"})
df = df[["image","labels","patient","age","sex","view"]]
test_ids = set(open(os.path.join(base, "test_list.txt")).read().split())
tv_ids = set(open(os.path.join(base, "train_val_list.txt")).read().split())
for f in FINDINGS:
    df[f] = df.labels.str.contains(f, regex=False).astype(int)
df = df[df.age <= 100].copy()
df["age_band4"] = pd.cut(df.age, [-1, 18, 40, 60, 200], labels=["0-18","18-40","40-60","60+"]).astype(str)
df["age_band5"] = pd.cut(df.age, [-1, 18, 40, 60, 80, 200], labels=["0-18","18-40","40-60","60-80","80+"]).astype(str)

rng = np.random.default_rng(SEED)
tv_patients = np.array(sorted(df[df.image.isin(tv_ids)].patient.unique()))
held_patients = set(rng.choice(tv_patients, size=int(round(0.10 * len(tv_patients))), replace=False).tolist())
df["set"] = np.where(df.image.isin(test_ids), "test",
             np.where(df.image.isin(tv_ids) & df.patient.isin(held_patients), "heldout", "unused"))
cohort = df[df.set != "unused"].reset_index(drop=True)
cohort["path"] = cohort.image.map(img_paths)
assert cohort.path.notna().all(), "some images not found"
assert not (set(cohort[cohort.set=="test"].patient) & set(cohort[cohort.set=="heldout"].patient))
cohort.drop(columns=["path"]).to_csv(f"{OUT}/cohort.csv", index=False)
run_info["n_test_images"] = int((cohort.set=="test").sum()); run_info["n_test_patients"] = int(cohort[cohort.set=="test"].patient.nunique())
run_info["n_heldout_images"] = int((cohort.set=="heldout").sum()); run_info["n_heldout_patients"] = len(held_patients)
print({k: run_info[k] for k in ["n_test_images","n_test_patients","n_heldout_images","n_heldout_patients"]})

In [ ]:
# 3. Eligible findings (identical rule to the registration)
test = cohort[cohort.set == "test"]
def eligible(data, col, groups):
    out = []
    for f in FINDINGS:
        if all(len(s := data[(data[col] == g) & (data[f] == 1)]) >= 30 and s.patient.nunique() >= 10 for g in groups):
            out.append(f)
    return out
SEX_ELIG = eligible(test, "sex", ["F", "M"]); AGE_ELIG = eligible(test, "age_band4", ["0-18", "18-40", "40-60", "60+"])
print("sex:", len(SEX_ELIG), "| age:", len(AGE_ELIG), AGE_ELIG)

In [ ]:
# 4. SFR-Embedding-Mistral embeddings for all prompt templates (separate process frees the GPU afterwards)
import gc
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from clear.concepts import ConceptBank
concepts_csv = hf_hub_download("peterhan91/CLEAR", "mimic_concepts.csv")
clip_pt = hf_hub_download("peterhan91/CLEAR", "concept_embeddings_368294.pt")
llm_pkl = hf_hub_download("peterhan91/CLEAR", "concepts_embeddings_sfr_mistral.pickle")
bank = ConceptBank.from_files(concepts_csv, clip_embeddings_path=clip_pt, llm_embeddings_path=llm_pkl, allow_unsafe_pickle=True)
C_llm = bank.ensure_llm_embeddings()
INSTR = "Instruct: Given medical radiology concepts, retrieve relevant passages and generate embeddings\nQuery: "
texts = {}
for t in TEMPLATES:
    texts[f"{t}_pos"] = TP_TXT[t]; texts[f"{t}_neg"] = TN_TXT[t]
texts["T1_pos_i"] = [INSTR + s for s in TP_TXT["T1"]]; texts["T1_neg_i"] = [INSTR + s for s in TN_TXT["T1"]]
json.dump(texts, open("/kaggle/working/sfr_in.json", "w"))
SFR_SCRIPT = r'''
import json, torch, numpy as np, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
d = json.load(open("/kaggle/working/sfr_in.json"))
tok = AutoTokenizer.from_pretrained("Salesforce/SFR-Embedding-Mistral")
mm = {i: "13GiB" for i in range(torch.cuda.device_count())}; mm["cpu"] = "24GiB"
llm = AutoModel.from_pretrained("Salesforce/SFR-Embedding-Mistral", dtype=torch.float16, device_map="auto", max_memory=mm).eval()
dev0 = next(llm.parameters()).device
def pool(h, m):
    if m[:, -1].sum() == m.shape[0]:
        return h[:, -1]
    return h[torch.arange(h.shape[0], device=h.device), m.sum(1) - 1]
@torch.no_grad()
def emb(ts):
    out = []
    for s in ts:
        e = tok(s, max_length=8192, truncation=True, return_tensors="pt").to(dev0)
        out.append(F.normalize(pool(llm(**e).last_hidden_state, e["attention_mask"]).float(), dim=-1).cpu())
    return torch.cat(out).numpy()
np.savez("/kaggle/working/sfr_out.npz", **{k: emb(v) for k, v in d.items()})
print("SFR embeddings done")
'''
open("/kaggle/working/sfr_embed.py", "w").write(SFR_SCRIPT)
r = subprocess.run([sys.executable, "/kaggle/working/sfr_embed.py"], capture_output=True, text=True)
print(r.stdout[-500:]); print(r.stderr[-2000:])
assert r.returncode == 0, "SFR embedding step failed"
SFR = dict(np.load("/kaggle/working/sfr_out.npz"))
pos_llm, neg_llm, pos_llm_i, neg_llm_i = SFR["T1_pos"], SFR["T1_neg"], SFR["T1_pos_i"], SFR["T1_neg_i"]
subprocess.run(["rm", "-rf", "/tmp/hf/hub/models--Salesforce--SFR-Embedding-Mistral"])
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 5. Load CLEAR and build the concept projection.
#    The authors' pipeline computes  normalise( (img @ norm(C_clip).T) @ C_llm ).
#    Because this is linear, it equals  normalise( img @ M ) with M = norm(C_clip).T @ C_llm,
#    which is computed once. Both forms are compared on real images in step 7.
from clear import load_pretrained, tokenize
dev = "cuda:0"
dev_cz = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
clear_model, clear_pre = load_pretrained(device=dev); clear_model.eval()
C_clip = torch.from_numpy(bank.ensure_clip_embeddings()).float()

# check that the released concept CLIP embeddings match CLEAR's text encoder (500 random concepts)
pick2 = np.random.default_rng(SEED+1).choice(bank.num_concepts, 500, replace=False)
with torch.no_grad():
    enc = F.normalize(clear_model.encode_text(tokenize([bank.concepts[i] for i in pick2]).to(dev)).float(), dim=-1).cpu()
cos_clip = (enc * F.normalize(C_clip[pick2], dim=-1)).sum(1)
run_info["concept_clip_check_cos_mean_min"] = [float(cos_clip.mean()), float(cos_clip.min())]
print("Released concept CLIP embeddings vs text encoder, cosine mean/min:", run_info["concept_clip_check_cos_mean_min"])
if cos_clip.min() < 0.999:
    print("Mismatch: recomputing all concept CLIP embeddings with the text encoder")
    chunks = []
    with torch.no_grad():
        for i in range(0, bank.num_concepts, 2048):
            chunks.append(clear_model.encode_text(tokenize(list(bank.concepts[i:i+2048])).to(dev)).float().cpu())
    C_clip = torch.cat(chunks); run_info["concept_clip_recomputed"] = True
else:
    run_info["concept_clip_recomputed"] = False

M = torch.zeros(C_clip.shape[1], C_llm.shape[1], dtype=torch.float64, device=dev)
C_llm_t = torch.from_numpy(C_llm)
for i in range(0, C_clip.shape[0], 8192):
    a = F.normalize(C_clip[i:i+8192].to(dev), dim=1).double()
    M += a.T @ C_llm_t[i:i+8192].to(dev).double()
M = M.float()
np.save(f"{OUT}/clear_concept_projection_M.npy", M.cpu().numpy())
pos_t = F.normalize(torch.from_numpy(pos_llm).to(dev), dim=-1); neg_t = F.normalize(torch.from_numpy(neg_llm).to(dev), dim=-1)
pos_ti = F.normalize(torch.from_numpy(pos_llm_i).to(dev), dim=-1); neg_ti = F.normalize(torch.from_numpy(neg_llm_i).to(dev), dim=-1)
with torch.no_grad():
    cpos = F.normalize(clear_model.encode_text(tokenize(POS).to(dev)).float(), dim=-1)
    cneg = F.normalize(clear_model.encode_text(tokenize(NEG).to(dev)).float(), dim=-1)
print("M:", tuple(M.shape))

In [ ]:
# 6. Load the 10 CheXzero checkpoints: from an attached Kaggle dataset if present, otherwise Google Drive
import gdown, clip
from torchvision.transforms import Compose, Normalize, Resize, InterpolationMode
def find_pts():
    hits = []
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if not d.startswith("images_")]
        hits += [os.path.join(root, f) for f in files if f.endswith(".pt")]
    return sorted(hits)
ckpts = find_pts()
if len(ckpts) < 10:
    os.makedirs("/kaggle/working/chexzero_weights", exist_ok=True)
    gdown.download_folder("https://drive.google.com/drive/folders/1makFLiEMbSleYltaRxw81aBhEDMpVwno",
                          output="/kaggle/working/chexzero_weights", quiet=True, remaining_ok=True)
    ckpts = sorted(glob.glob("/kaggle/working/chexzero_weights/**/*.pt", recursive=True))
assert len(ckpts) == 10, f"expected 10 CheXzero checkpoints, found {len(ckpts)}: {ckpts}"
print("CheXzero checkpoints from:", os.path.dirname(ckpts[0]))
run_info["chexzero_checkpoints"] = [[os.path.basename(c), os.path.getsize(c)] for c in ckpts]
cz_models, cz_text = [], []
for c in ckpts:
    m, _ = clip.load("ViT-B/32", device="cpu", jit=False)
    m.load_state_dict(torch.load(c, map_location="cpu")); m = m.to(dev_cz).eval()
    with torch.no_grad():
        tp = F.normalize(m.encode_text(clip.tokenize(POS, context_length=77).to(dev_cz)).float(), dim=-1)
        tn = F.normalize(m.encode_text(clip.tokenize(NEG, context_length=77).to(dev_cz)).float(), dim=-1)
    cz_models.append(m); cz_text.append((tp, tn))
print("CheXzero checkpoints loaded:", len(cz_models), "| GPU GB in use:", [round(torch.cuda.memory_allocated(i)/1e9, 2) for i in range(torch.cuda.device_count())])

In [ ]:
# 7. Text embeddings for every template: CheXzero (each checkpoint), CLEAR text encoder, SFR (concept route)
with torch.no_grad():
    czT = [{t: (F.normalize(m.encode_text(clip.tokenize(TP_TXT[t], context_length=77).to(dev_cz)).float(), dim=-1),
                F.normalize(m.encode_text(clip.tokenize(TN_TXT[t], context_length=77).to(dev_cz)).float(), dim=-1))
            for t in TEMPLATES} for m in cz_models]
    clT = {t: (F.normalize(clear_model.encode_text(tokenize(TP_TXT[t]).to(dev)).float(), dim=-1),
               F.normalize(clear_model.encode_text(tokenize(TN_TXT[t]).to(dev)).float(), dim=-1)) for t in TEMPLATES}
sfT = {t: (F.normalize(torch.from_numpy(SFR[f"{t}_pos"]).to(dev), dim=-1), F.normalize(torch.from_numpy(SFR[f"{t}_neg"]).to(dev), dim=-1))
       for t in TEMPLATES}

In [ ]:
# 8. One pass over all cohort images: scores for every model x template; keep CLEAR and CheXzero (checkpoint 1) features
from PIL import Image
from torch.utils.data import Dataset, DataLoader
cz_tf = Compose([Normalize((101.48761,)*3, (83.43944,)*3), Resize(224, interpolation=InterpolationMode.BICUBIC)])
class DS(Dataset):
    def __init__(self, paths): self.p = list(paths)
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        im = Image.open(self.p[i])
        g = np.asarray(im.convert("L").resize((320, 320), Image.BICUBIC), dtype=np.float32)
        return cz_tf(torch.from_numpy(np.repeat(g[None], 3, axis=0))), clear_pre(im.convert("RGB"))
def pair_prob(f, tp, tn):
    p, n = f @ tp.T, f @ tn.T
    return torch.exp(p) / (torch.exp(p) + torch.exp(n))
BS = 32
dl = DataLoader(DS(cohort.path), batch_size=BS, num_workers=4, pin_memory=True)
SC = {(m, t): [] for m in ["chexzero", "clear_direct", "clear_concept"] for t in TEMPLATES}
SC[("clear_concept_instr", "T1")] = []
FEAT_CL, FEAT_CZ1 = [], []
tm = time.time()
with torch.no_grad():
    for b, (x_cz, x_cl) in enumerate(dl):
        x_cz, x_cl = x_cz.to(dev_cz, non_blocking=True), x_cl.to(dev, non_blocking=True)
        acc = {t: 0 for t in TEMPLATES}
        for k, m in enumerate(cz_models):
            f = F.normalize(m.encode_image(x_cz).float(), dim=-1)
            if k == 0: FEAT_CZ1.append(f.half().cpu())
            for t in TEMPLATES: acc[t] = acc[t] + pair_prob(f, *czT[k][t])
        for t in TEMPLATES: SC[("chexzero", t)].append((acc[t] / len(cz_models)).cpu())
        fi = F.normalize(clear_model.encode_image(x_cl).float(), dim=-1)
        FEAT_CL.append(fi.cpu())
        rep = F.normalize(fi @ M, dim=-1)
        for t in TEMPLATES:
            SC[("clear_direct", t)].append(pair_prob(fi, *clT[t]).cpu())
            SC[("clear_concept", t)].append(pair_prob(rep, *sfT[t]).cpu())
        SC[("clear_concept_instr", "T1")].append(pair_prob(rep, pos_ti, neg_ti).cpu())
        if b % 100 == 0: print(f"{min((b+1)*BS, len(cohort))}/{len(cohort)} images, {round((time.time()-tm)/60,1)} min", flush=True)
SC = {k: torch.cat(v).double().numpy() for k, v in SC.items()}
FEAT_CL = torch.cat(FEAT_CL); FEAT_CZ1 = torch.cat(FEAT_CZ1).float()
run_info["min_scoring"] = round((time.time()-tm)/60, 1)
np.savez_compressed(f"{OUT}/scores_all_templates.npz", **{f"{m}__{t}": v.astype(np.float32) for (m, t), v in SC.items()})

In [ ]:
# 9. Bootstrap machinery (patient level, 1,000 resamples, seed 2026), identical to the registered analysis
is_test = (cohort.set == "test").values; is_held = (cohort.set == "heldout").values
T = cohort[is_test].reset_index(drop=True)
Y = T[FINDINGS].values.astype(np.float64)
pat_codes, pat_idx = np.unique(T.patient.values, return_inverse=True); NP = len(pat_codes)
rng = np.random.default_rng(SEED)
WB = [np.bincount(rng.integers(0, NP, NP), minlength=NP).astype(np.float64) for _ in range(1000)]
G = {"sex_F": (T.sex == "F").values, "sex_M": (T.sex == "M").values}
for a in ["0-18", "18-40", "40-60", "60+"]: G[f"age_{a}"] = (T.age_band4 == a).values
class Cell:
    def __init__(self, s, y, pidx):
        o = np.argsort(s, kind="mergesort"); s = s[o]
        self.y, self.pidx = y[o], pidx[o]
        self.starts = np.flatnonzero(np.r_[True, s[1:] != s[:-1]])
    def auc(self, w=None):
        wi = 1.0 if w is None else w[self.pidx]
        bp = np.add.reduceat(wi * self.y, self.starts); bn = np.add.reduceat(wi * (1 - self.y), self.starts)
        P, N = bp.sum(), bn.sum()
        if P == 0 or N == 0: return np.nan
        return float((bp * (np.cumsum(bn) - bn + 0.5 * bn)).sum() / (P * N))
def ci(a, lvl=0.95):
    a = a[~np.isnan(a)]
    if a.size == 0: return (float("nan"), float("nan"))
    return float(np.quantile(a, (1-lvl)/2)), float(np.quantile(a, 1-(1-lvl)/2))
def auc_table(s_test):
    """point and bootstrap AUROC per (group, finding); groups include 'all'."""
    groups = {"all": np.ones(len(T), bool), **G}
    pt, bt = {}, {}
    for g, m in groups.items():
        idx = np.flatnonzero(m)
        for j, f in enumerate(FINDINGS):
            c = Cell(s_test[idx, j], Y[idx, j], pat_idx[idx])
            pt[(g, f)] = c.auc(); bt[(g, f)] = np.array([c.auc(w) for w in WB])
    return pt, bt
def macro(pt, bt, g, flist):
    return float(np.nanmean([pt[(g, f)] for f in flist])), np.nanmean(np.array([bt[(g, f)] for f in flist]), axis=0)
def gaps(pt, bt):
    out = {}
    pF, bF = macro(pt, bt, "sex_F", SEX_ELIG); pM, bM = macro(pt, bt, "sex_M", SEX_ELIG)
    out["sex_F_minus_M"] = (pF - pM, ci(bF - bM, 0.95))
    pR, bR = macro(pt, bt, "age_40-60", AGE_ELIG)
    for a in ["0-18", "18-40", "60+"]:
        p, b = macro(pt, bt, f"age_{a}", AGE_ELIG); out[f"age_{a}_minus_40-60"] = (p - pR, ci(b - bR, 1 - 0.05/3))
    return out

In [ ]:
# 10. A2: prompt robustness
A2 = {}
for (m, t), s in SC.items():
    if m == "clear_concept_instr": continue
    pt, bt = auc_table(s[is_test])
    p, b = macro(pt, bt, "all", FINDINGS)
    A2[(m, t)] = {"macro_all": (p, ci(b)), "gaps": gaps(pt, bt), "per_finding": {f: round(pt[("all", f)], 3) for f in FINDINGS}}
    print(m, t, "macro", round(p, 3), flush=True)
def verdict(g):
    p, (lo, hi) = g
    return "gap+" if lo > 0 else ("gap-" if hi < 0 else "no gap")
robust = {}
for m in ["chexzero", "clear_concept", "clear_direct"]:
    ref = {k: verdict(v) for k, v in A2[(m, "T1")]["gaps"].items()}
    diffs = [(t, k, verdict(v)) for t in ["T2", "T3", "T4"] for k, v in A2[(m, t)]["gaps"].items() if verdict(v) != ref[k]]
    robust[m] = {"T1_verdicts": ref, "differences": diffs, "prompt_robust": len(diffs) == 0}

In [ ]:
# 11. A1: auditability of CLEAR concept explanations (template T1, plain prompts)
import re
from sklearn.metrics import roc_curve
s_cc = SC[("clear_concept", "T1")]
thr = {}
for j, f in enumerate(FINDINGS):
    yh = cohort.loc[is_held, f].values; fpr, tpr, th = roc_curve(yh, s_cc[is_held, j]); thr[f] = float(th[np.argmax(tpr - fpr)])
pred = np.stack([s_cc[is_test, j] >= thr[f] for j, f in enumerate(FINDINGS)], 1)
cat = np.where(pred & (Y == 1), "TP", np.where(~pred & (Y == 1), "FN", np.where(pred & (Y == 0), "FP", "TN")))

P1 = F.normalize(torch.from_numpy(SFR["T1_pos"]), dim=-1).numpy(); N1 = F.normalize(torch.from_numpy(SFR["T1_neg"]), dim=-1).numpy()
Wc = torch.from_numpy((C_llm @ (P1 - N1).T).astype(np.float32)).to(dev)            # concepts x findings
Cn = F.normalize(C_clip, dim=1).to(dev)                                            # concepts x 768
REL = torch.stack([torch.tensor([bool(re.search(RELEVANCE[f], c, re.I)) for c in bank.concepts]) for f in FINDINGS], 1).to(dev)
run_info["relevant_concepts_per_finding"] = {f: int(REL[:, j].sum()) for j, f in enumerate(FINDINGS)}
print("relevant concepts:", run_info["relevant_concepts_per_finding"])

FT = FEAT_CL[torch.from_numpy(is_test)]; FH = FEAT_CL[torch.from_numpy(is_held)]
rel_share = np.zeros((len(T), len(FINDINGS)), np.float32)
QUAL = ["Cardiomegaly", "Emphysema", "Hernia"]
qsum = {(f, c): torch.zeros(Cn.shape[0], device=dev, dtype=torch.float64) for f in QUAL for c in ["TP", "FN"]}
qn = {(f, c): 0 for f in QUAL for c in ["TP", "FN"]}
tm = time.time()
with torch.no_grad():
    for i in range(0, len(T), 32):
        s = FT[i:i+32].to(dev) @ Cn.T                                              # B x concepts
        for j, f in enumerate(FINDINGS):
            contrib = s * Wc[:, j]
            top = contrib.topk(10, dim=1).indices
            rel_share[i:i+32, j] = REL[top, j].float().mean(1).cpu().numpy()
            if f in QUAL:
                for c in ["TP", "FN"]:
                    sel = torch.from_numpy(cat[i:i+32, j] == c).to(dev)
                    if sel.any(): qsum[(f, c)] += contrib[sel].double().sum(0); qn[(f, c)] += int(sel.sum())
print("attribution pass", round((time.time()-tm)/60, 1), "min")

# 1. relevance by outcome category
REL_BY_CAT = {f: {c: (float(rel_share[cat[:, j] == c, j].mean()) if (cat[:, j] == c).any() else None) for c in ["TP", "FN", "FP"]}
              for j, f in enumerate(FINDINGS)}
# 2. explanation fairness among true positives
def rel_group(j, gmask, w=None):
    sel = (cat[:, j] == "TP") & gmask
    if not sel.any(): return np.nan
    wi = np.ones(sel.sum()) if w is None else w[pat_idx[sel]]
    return float((wi * rel_share[sel, j]).sum() / wi.sum()) if wi.sum() > 0 else np.nan
def rel_macro(gname, flist, w=None):
    return np.nanmean([rel_group(FINDINGS.index(f), G[gname], w) for f in flist])
EXPL_FAIR = {}
for name, a, b_, flist, lvl in [("sex_F_minus_M", "sex_F", "sex_M", SEX_ELIG, 0.95)] + \
        [(f"age_{a}_minus_40-60", f"age_{a}", "age_40-60", AGE_ELIG, 1 - 0.05/3) for a in ["0-18", "18-40", "60+"]]:
    p = rel_macro(a, flist) - rel_macro(b_, flist)
    bs = np.array([rel_macro(a, flist, w) - rel_macro(b_, flist, w) for w in WB])
    EXPL_FAIR[name] = (float(p), ci(bs, lvl))
# 3. single-concept probe: select on held-out, evaluate on test
def gpu_auc(X, y):
    r = X.argsort(0).argsort(0).double() + 1; P = y.sum(); N = len(y) - P
    return (r[y.bool()].sum(0) - P * (P + 1) / 2) / (P * N)
PROBE = {}
with torch.no_grad():
    for j, f in enumerate(FINDINGS):
        R = torch.nonzero(REL[:, j]).flatten()
        if len(R) == 0: PROBE[f] = None; continue
        yh = torch.from_numpy(cohort.loc[is_held, f].values).to(dev)
        best, best_auc = None, -1
        for k in range(0, len(R), 2048):
            idx = R[k:k+2048]; a = gpu_auc(FH.to(dev) @ Cn[idx].T, yh)
            v, p_ = a.max(0)
            if v.item() > best_auc: best_auc, best = v.item(), idx[p_].item()
        s_t = (FT.to(dev) @ Cn[best]).double().cpu().numpy()
        c = Cell(s_t, Y[:, j], pat_idx); bs = np.array([c.auc(w) for w in WB])
        PROBE[f] = {"concept": bank.concepts[best], "heldout_auc": round(best_auc, 3), "test_auc": (round(c.auc(), 3), ci(bs)),
                    "clear_concept_auc": A2[("clear_concept", "T1")]["per_finding"][f], "clear_direct_auc": A2[("clear_direct", "T1")]["per_finding"][f],
                    "chexzero_auc": A2[("chexzero", "T1")]["per_finding"][f]}
# 4. representation spread
sel = np.random.default_rng(SEED).choice(len(T), min(2000, len(T)), replace=False)
def spread(X):
    X = F.normalize(X.double(), dim=1); S = X @ X.T; n = len(X)
    mc = float((S.sum() - n) / (n * (n - 1)))
    ev = torch.linalg.eigvalsh(torch.cov((X - X.mean(0)).T)).clamp_min(0)
    return {"mean_pairwise_cos": round(mc, 4), "participation_ratio": round(float(ev.sum()**2 / (ev**2).sum()), 2)}
with torch.no_grad():
    SPREAD = {"clear_image": spread(FT[sel]), "clear_concept_space": spread((FT[sel].to(dev) @ M).cpu()), "chexzero_ckpt1": spread(FEAT_CZ1[torch.from_numpy(is_test)][sel])}
# qualitative top concepts
QUAL_TOP = {}
for (f, c), v in qsum.items():
    if qn[(f, c)] == 0: continue
    top = (v / qn[(f, c)]).topk(15).indices.cpu().numpy()
    QUAL_TOP[f"{f}_{c}"] = [bank.concepts[i] for i in top]

In [ ]:
# 12. Save and print the summary
def jsonable(o):
    if isinstance(o, dict): return {("|".join(k) if isinstance(k, tuple) else str(k)): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(x) for x in o]
    if isinstance(o, (np.floating, np.integer)): return o.item()
    return o
json.dump(jsonable({"run_info": run_info, "A2": A2, "A2_robust": robust, "A1_relevance_by_category": REL_BY_CAT,
                    "A1_explanation_fairness": EXPL_FAIR, "A1_probe": PROBE, "A1_spread": SPREAD, "A1_qualitative": QUAL_TOP,
                    "thresholds": thr}), open(f"{OUT}/amend1_results.json", "w"), indent=1)
!cd /kaggle/working && rm -f amend1_outputs.zip && zip -qr amend1_outputs.zip amend1_out -x "*.npz"
print("===== AMENDMENT SUMMARY =====")
print("A2 prompt robustness: macro-AUROC (all 14) and gaps [estimate (interval)]")
for m in ["chexzero", "clear_concept", "clear_direct"]:
    for t in TEMPLATES:
        r = A2[(m, t)]; p, (lo, hi) = r["macro_all"]
        g = " | ".join(f"{k.replace('_minus_', '-')}: {v[0]:+.3f} ({v[1][0]:+.3f},{v[1][1]:+.3f})" for k, v in r["gaps"].items())
        print(f"  {m:<14} {t} macro {p:.3f} ({lo:.3f}-{hi:.3f}) | {g}")
    print(f"  -> {m} prompt-robust: {robust[m]['prompt_robust']} | differences: {robust[m]['differences']}")
print("A1.1 relevance of top-10 concepts (TP / FN / FP):")
for f, v in REL_BY_CAT.items():
    print(f"  {f:<20} " + " / ".join("NA" if v[c] is None else f"{v[c]:.2f}" for c in ["TP", "FN", "FP"]))
print("A1.2 explanation fairness in true positives (relevance difference):")
for k, (p, (lo, hi)) in EXPL_FAIR.items(): print(f"  {k:<22} {p:+.3f} ({lo:+.3f} to {hi:+.3f})")
print("A1.3 single-concept probe (concept: test AUROC vs concept route / direct / CheXzero):")
for f, v in PROBE.items():
    if v: print(f"  {f:<20} '{v['concept'][:45]}': {v['test_auc'][0]:.3f} ({v['test_auc'][1][0]:.3f}-{v['test_auc'][1][1]:.3f}) vs {v['clear_concept_auc']:.3f} / {v['clear_direct_auc']:.3f} / {v['chexzero_auc']:.3f}")
print("A1.4 representation spread:", SPREAD)
print("A1 qualitative top concepts:")
for k, v in QUAL_TOP.items(): print(f"  {k}: {v[:8]}")
print("relevant concepts per finding:", run_info["relevant_concepts_per_finding"])
print("minutes:", round((time.time()-T0)/60, 1))
print("===== END =====")